<a href="https://colab.research.google.com/github/kyuuf/Youtube_Comment_Crawler/blob/master/%EA%B0%90%EC%84%B1_%EB%B6%84%EC%84%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[구글 코랩용] 나만의 감성 사전 분석기

1. KNU 감성사전 리포지토리 다운로드

In [ ]:
import os
import json
import pandas as pd
import glob

print("1. 감성 사전 다운로드를 시작합니다...")
if os.path.exists('KnuSentiLex'):
    !rm -rf KnuSentiLex  # 기존 폴더 삭제 (충돌 방지)

!git clone https://github.com/park1200656/KnuSentiLex.git

2. 사전 파일 자동 탐색 및 로드


In [ ]:
# 깃허브 폴더 구조가 바뀌어도 자동으로 파일을 찾습니다.
search_path = 'KnuSentiLex/**/SentiWord_info.json'
found_files = glob.glob(search_path, recursive=True)

if found_files:
    json_path = found_files[0]
    print(f"2. 사전 파일을 찾았습니다: {json_path}")

    with open(json_path, encoding='utf-8-sig', mode='r') as f:
        senti_data = json.load(f)
        print(f"   -> 사전 로드 완료! (총 {len(senti_data)}개 단어)")
else:
    print("❌ 오류: 'SentiWord_info.json' 파일을 찾을 수 없습니다. 다운로드에 실패했을 수 있습니다.")
    senti_data = [] # 빈 리스트로 초기화


3. 감성 분석기 클래스 정의



In [ ]:
class KnuSL:
    def __init__(self, data):
        self.data = data

    def calculate_score(self, sentence):
        if not self.data or not isinstance(sentence, str):
            return 0, [] # 데이터 없으면 0점

        score = 0
        matches = []
        for entry in self.data:
            word = entry['word']
            polarity = int(entry['polarity'])
            if word in sentence:
                score += polarity
                matches.append(word)
        return score, matches

    def get_sentiment(self, score):
        if score >= 1: return "긍정"
        elif score <= -1: return "부정"
        else: return "중립"

4.  데이터 분석 실행

In [ ]:
filename = '한글 손흥민 댓글 합친본.csv' # 업로드한 파일명 확인

if not senti_data:
    print("⛔ 사전 데이터가 없어 분석을 중단합니다.")
else:
    # 파일 읽기
    try:
        try:
            df = pd.read_csv(filename, encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(filename, encoding='cp949')

        if 'text' in df.columns:
            print(f"\n3. '{filename}' 분석 시작... (총 {len(df)}건)")
            knu = KnuSL(senti_data)

            # 분석 수행
            results = [knu.calculate_score(text) for text in df['text']]

            df['score'] = [r[0] for r in results]
            df['matches'] = [r[1] for r in results]
            df['sentiment'] = df['score'].apply(knu.get_sentiment)

            # 결과 저장
            output_name = 'son_sentiment_result.csv'
            df.to_csv(output_name, index=False, encoding='utf-8-sig')

            print(f"\n✅ 분석 완료! 결과가 '{output_name}' 파일로 저장되었습니다.")
            print("\n[결과 미리보기]")
            display(df[['text', 'sentiment', 'score', 'matches']].head(5))

        else:
            print("❌ 오류: CSV 파일에 'text' 컬럼이 없습니다.")

    except FileNotFoundError:
        print(f"❌ 오류: '{filename}' 파일이 없습니다. 코랩 왼쪽에 파일을 업로드했는지 확인해주세요.")

In [ ]:
# ==========================================
# 1. 환경 설정 및 데이터 다운로드 (필수)
# ==========================================
print("1. 필수 라이브러리 설치 및 사전 다운로드 중...")
!pip install -q konlpy # 형태소 분석기 설치
import os
import glob
import json
import pandas as pd
import numpy as np
import re
from konlpy.tag import Okt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# 기존 폴더 삭제 후 다시 다운로드 (경로 에러 방지)
if os.path.exists('KnuSentiLex'):
    !rm -rf KnuSentiLex
!git clone https://github.com/park1200656/KnuSentiLex.git

# ==========================================
# 2. 사전 파일 자동 탐색 및 로드
# ==========================================
# 파일이 어느 폴더에 숨어있든 찾아냅니다.
search_path = 'KnuSentiLex/**/SentiWord_info.json'
found_files = glob.glob(search_path, recursive=True)

if not found_files:
    print("❌ 오류: 사전 파일을 찾을 수 없습니다. 다시 실행해 주세요.")
else:
    json_path = found_files[0]
    print(f"2. 사전 파일 연결 성공: {json_path}")

    with open(json_path, encoding='utf-8-sig', mode='r') as f:
        senti_data = json.load(f)

    # 감성 점수 계산 함수 (사전 기반)
    def get_sentiment_score(text):
        if not isinstance(text, str): return 0
        score = 0
        # 간단한 매칭 (속도 최적화)
        for entry in senti_data:
            if entry['word'] in text:
                score += int(entry['polarity'])
        return score

    # ==========================================
    # 3. 데이터 로드 및 전처리
    # ==========================================
    filename = '영어(번역)종합본123.csv'

    try:
        try:
            df = pd.read_csv(filename, encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(filename, encoding='cp949')

        if 'text' in df.columns:
            print(f"3. 데이터 로드 완료: {len(df)}건")
            print("   형태소 분석 및 토픽 모델링 시작... (1~2분 소요될 수 있습니다)")

            # 형태소 분석기 초기화
            okt = Okt()

            # 불용어 리스트
            stop_words = [
                '이', '그', '저', '것', '수', '등', '들', '좀', '잘', '걍', '죠', '임',
                '거', '때', '때문', '너무', '정말', '진짜', '에서', '에게', '으로', '하다',
                'ㅅㅂ', '이다', '있다', '없다', '같다', '그리고', '하지만', '그냥', '또',
                '존나', '왜', '뭐', '이거', '저거', '영상', '댓글', '유튜브', '사람','손흥민','쏘니','소니','흥미니'
            ]

            # 텍스트 전처리 함수 (명사 추출)
            def tokenizer(text):
                if not isinstance(text, str): return []
                text = re.sub(r'[^가-힣\s]', '', text) # 한글만 남김
                nouns = okt.nouns(text) # 명사 추출
                return [n for n in nouns if len(n) > 1 and n not in stop_words] # 2글자 이상만 사용, 불용어 제거

            # ==========================================
            # 4. 토픽 모델링 (LDA)
            # ==========================================
            # 단어 벡터화
            vectorizer = CountVectorizer(tokenizer=tokenizer, max_features=1000, max_df=0.5)
            dtm = vectorizer.fit_transform(df['text'].astype(str))

            # LDA 모델 학습 (토픽 4개)
            lda = LatentDirichletAllocation(n_components=4, random_state=42)
            lda_output = lda.fit_transform(dtm)

            # 각 문서의 주된 토픽 결정
            df['Topic'] = np.argmax(lda_output, axis=1)

            # ==========================================
            # 5. 토픽별 감성 분석 결합
            # ==========================================
            print("   감성 점수 계산 중...")
            df['Score'] = df['text'].apply(get_sentiment_score)

            # 토픽별 평균 감성 점수 계산
            topic_summary = df.groupby('Topic')['Score'].mean()

            # 결과 출력
            print("\n" + "="*50)
            print("[토픽 모델링 + 감성 분석 결과]")
            print("="*50)

            feature_names = vectorizer.get_feature_names_out()

            for topic_idx, topic in enumerate(lda.components_):
                # 키워드 상위 5개
                top_features_ind = topic.argsort()[:-6:-1]
                top_features = [feature_names[i] for i in top_features_ind]

                avg_score = topic_summary[topic_idx]
                sentiment_label = "긍정" if avg_score > 0.5 else ("부정" if avg_score < -0.5 else "중립")

                print(f"▶ Topic {topic_idx} 주요 키워드: {top_features}")
                print(f"   ㄴ 평균 감성 점수: {avg_score:.2f} ({sentiment_label})")
                print("-" * 50)

            # 결과 저장
            output_file = 'son_topic_sentiment_result.csv'
            df.to_csv(output_file, index=False, encoding='utf-8-sig')
            print(f"\n✅ 상세 결과가 '{output_file}'로 저장되었습니다.")

        else:
            print("❌ 오류: CSV 파일 내에 'text' 컬럼이 없습니다.")

    except FileNotFoundError:
        print(f"❌ 오류: '{filename}' 파일을 찾을 수 없습니다. 파일을 업로드했는지 확인해주세요.")

1. 필수 라이브러리 설치 및 사전 다운로드 중...
Cloning into 'KnuSentiLex'...
remote: Enumerating objects: 45, done.
remote: Total 45 (delta 0), reused 0 (delta 0), pack-reused 45 (from 1)
Receiving objects: 100% (45/45), 380.68 KiB | 6.80 MiB/s, done.
Resolving deltas: 100% (15/15), done.
2. 사전 파일 연결 성공: KnuSentiLex/KnuSentiLex/data/SentiWord_info.json
3. 데이터 로드 완료: 11167건
   형태소 분석 및 토픽 모델링 시작... (1~2분 소요될 수 있습니다)


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


   감성 점수 계산 중...

[토픽 모델링 + 감성 분석 결과]
▶ Topic 0 주요 키워드: ['축구', '로스앤젤레스', '클럽', '선수', '우리']
   ㄴ 평균 감성 점수: 0.46 (중립)
--------------------------------------------------
▶ Topic 1 주요 키워드: ['리그', '메이저리그사커', '선수', '프리미어리그', '수비']
   ㄴ 평균 감성 점수: 0.01 (중립)
--------------------------------------------------
▶ Topic 2 주요 키워드: ['토트넘', '프리킥', '구독', '모습', '클래스']
   ㄴ 평균 감성 점수: -0.10 (중립)
--------------------------------------------------
▶ Topic 3 주요 키워드: ['메이저리그사커', '선수', '미국', '부앙', '역시']
   ㄴ 평균 감성 점수: 0.22 (중립)
--------------------------------------------------

✅ 상세 결과가 'son_topic_sentiment_result.csv'로 저장되었습니다.
